<a href="https://colab.research.google.com/github/TWalkerSMCM/icsa-scraper/blob/main/examples/skipper-elo.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# icsa-scraper · Fleet Race Skipper Elo

Rate individual skippers from per-race fleet finishes. Each race is scored as
pairwise win/loss/tie against every other boat in that division's fleet (no
margin of victory) — a chess-style Elo update, applied simultaneously to every
boat in the race using **pre-race** ratings.

This reproduces the core of the [analytics webapp](https://collegeanalytics.goforthom.as)'s
"Fleet Race Skipper Elo" feature, whose authoritative implementation lives at
`analytics/skipper_elo/engine.py` in the `icsa-tools` monorepo (339 lines). This
notebook keeps the rating math exactly as written there — constants, K-scaling,
expected-score formula, chronological ordering, grade derivation — but drops the
production-only bookkeeping (weekly rank-timeline snapshots, cross-division A/B
calibration, DynamoDB-shaped output) to keep the loop readable end to end.

## Install

In [ ]:
# Fresh runtimes (Colab) need the library. Guard on the distribution name so
# re-running locally can't clobber an editable dev install; to force-update a
# cached Colab runtime, restart it (or run the %pip line with --force-reinstall).
from importlib.metadata import PackageNotFoundError, version

try:
    version("icsa-scraper")
except PackageNotFoundError:
    %pip install -q "icsa-scraper[fetch] @ git+https://github.com/TWalkerSMCM/icsa-scraper"

## Optional: persist the cache on Colab

`scraper.load()` caches every fetched page under `./.scraper_cache`, so re-running a cell (or the whole notebook) after the first pass is instant. Colab wipes that folder on runtime reset — mount Drive and point the cache there first if you want it to survive. Set `SCRAPER_CACHE_DIR` **before** importing `scraper`:

```python
# from google.colab import drive
# drive.mount('/content/drive')
# import os
# os.environ['SCRAPER_CACHE_DIR'] = '/content/drive/MyDrive/icsa_cache'
```

## The rating engine's constants

Straight from `engine.py`: a regatta's `type` string maps to a **grade**
0 (nationals) … 5 (lowest); the grade sets the K-factor. A handful of types
never count at all (scrimmages, promotional/in-conference events). Only
dinghy fleet racing counts — keelboat/offshore/singlehanded/match-race boats
are excluded, since those classes aren't comparable to standard fleet racing.

`PROVISIONAL_RACES` doesn't drop a skipper — it just flags a rating built on
too few races as low-confidence, same as the website does.

In [ ]:
# regatta.type -> grade (0..5). Types absent here default to DEFAULT_GRADE.
GRADE_BY_TYPE = {
    "National Championship Finals Regatta": 0,
    "Showcase Regatta": 0,
    "National Championship Semifina Regatta": 1,
    "Conference Championship Regatta": 2,
    "Cross Regional Regatta": 2,
    "Regional Regatta": 3,
    "Fundamental Regatta": 4,
}
DEFAULT_GRADE = 4

# Types that never count toward ratings
EXCLUDE_TYPES = {
    "Scrimmage Regatta",
    "NOT ALLOWED  Promotional Regatta",
    "NOT ALLOWED  In-Conference Regatta",
}

# Keelboat / offshore / singlehanded / match-race classes excluded from fleet Elo
EXCLUDED_BOATS = (
    "J-70",
    "J-22",
    "Catalina 37",
    "Navy 44",
    "Colgate 26",
    "Sonar",
    "Tartan 10",
    "Melges 24",
    "Flying Scot",
    "Rhodes",
    "Laser",
    "Radial",
    "Match Race 3-4",
)

K_BY_GRADE = {0: 32, 1: 28, 2: 24, 3: 20, 4: 16, 5: 12}
BASE_RATING = 1500.0
PROVISIONAL_RACES = 20  # below this many races, a rating is flagged low-confidence

# The full academic year: ratings carry from fall into spring.
SEASONS = ["f25", "s26"]
PARTICIPANT = "coed"  # or "women"

## The rating formula

For a race with `n` boats, `k_race = K_BY_GRADE[grade] / (n - 1)` — the
K-factor is split across the `n - 1` pairwise comparisons a boat is party to,
so a 20-boat fleet race doesn't move ratings 20x harder than a match race.

For every pair `(i, j)` in the race, using **ratings as of before this race**:

```
expected_i = 1 / (1 + 10 ** ((R_j - R_i) / 400))
actual_i   = 1 if place_i < place_j else (0 if place_i > place_j else 0.5)
delta_i    = k_race * (actual_i - expected_i)
```

All pairwise deltas for a race are summed and applied to each boat's rating
**after** the whole race is scored — so results are simultaneous, not
order-dependent on how the pairs happen to be enumerated.

## Load a season

`.fleet()` narrows the dataset to fleet-scoring regattas only; `sailor_races`
is the RP-joined per-boat-role finish log. `progress=True` shows a
per-regatta progress bar while it runs. We keep skippers only, drop excluded
boat classes and non-counting regatta types, and grade what's left.

In [ ]:
import scraper

data = scraper.load(SEASONS, sailors=True, progress=True).fleet()
print(len(data.regattas), "fleet regattas |", len(data.sailor_races), "sailor-race rows")

In [ ]:
import pandas as pd

# Truncate start_time to the date, matching the engine's _parse_date ([:10]) —
# ratings tick by regatta day, not wall-clock start.
races = data.sailor_races_frame()
races["start_time"] = races["start_time"].dt.tz_localize(None).dt.normalize()

races = races[
    (races["boat_role"] == "skipper")
    & (races["participant"] == PARTICIPANT)
    & (~races["regatta_type"].isin(EXCLUDE_TYPES))
    & (~races["boat"].isin(EXCLUDED_BOATS))
].copy()
races["grade"] = races["regatta_type"].map(GRADE_BY_TYPE).fillna(DEFAULT_GRADE).astype(int)
print(len(races), "skipper-race rows after filtering")
races.head()

## Order races chronologically and size each fleet

The engine orders by `(start_time, regatta_slug, division, race_num)` — races
must be visited in real-world order since ratings compound. Fleet size `n`
(boats in that division/race) comes straight from counting rows in the group;
races with fewer than 2 boats can't produce a comparison and are dropped.

In [ ]:
races = races.sort_values(["start_time", "regatta_slug", "division", "race_num"]).reset_index(
    drop=True
)
races["fleet_size"] = races.groupby(["regatta_slug", "division", "race_num"])["place"].transform(
    "size"
)
races = races[races["fleet_size"] >= 2]

## The rating loop

One pass over the races, in order. Within a race, every pairwise rating uses
the *pre-race* ratings snapshot (`pre`) — this is what makes the update
simultaneous rather than sequential.

In [ ]:
from collections import defaultdict

ratings = defaultdict(lambda: BASE_RATING)
races_count = defaultdict(int)
history = []  # per-race post-rating, for the trajectory chart

race_key = ["start_time", "regatta_slug", "division", "race_num"]
for key, g in races.groupby(race_key, sort=False):
    start_time, regatta_slug, division, race_num = key
    n = len(g)
    grade = int(g["grade"].iloc[0])
    k_race = K_BY_GRADE.get(grade, 16) / (n - 1)

    boats = list(zip(g["sailor_slug"], g["place"]))
    pre = {sid: ratings[sid] for sid, _ in boats}  # pre-race snapshot
    deltas = defaultdict(float)

    for i in range(n):
        si, pi = boats[i]
        Ri = pre[si]
        for j in range(i + 1, n):
            sj, pj = boats[j]
            Rj = pre[sj]
            exp_i = 1.0 / (1.0 + 10 ** ((Rj - Ri) / 400.0))
            act_i = 1.0 if pi < pj else (0.0 if pi > pj else 0.5)
            deltas[si] += k_race * (act_i - exp_i)
            deltas[sj] += k_race * ((1.0 - act_i) - (1.0 - exp_i))

    for sid, _ in boats:
        ratings[sid] += deltas[sid]
        races_count[sid] += 1
        history.append(
            {
                "start_time": start_time,
                "regatta_slug": regatta_slug,
                "sailor_slug": sid,
                "elo": ratings[sid],
                "race_seq": races_count[sid],
            }
        )

print(len(ratings), "skippers rated over", len(history), "boat-races")

## Rank the skippers

In [ ]:
names = races.drop_duplicates("sailor_slug").set_index("sailor_slug")[
    ["sailor_name", "school_slug"]
]

rankings = pd.DataFrame(
    {
        "sailor_slug": list(ratings.keys()),
        "elo": [round(v, 1) for v in ratings.values()],
        "races": [races_count[s] for s in ratings.keys()],
    }
).merge(names, left_on="sailor_slug", right_index=True)

rankings["provisional"] = rankings["races"] < PROVISIONAL_RACES
rankings = rankings.sort_values("elo", ascending=False).reset_index(drop=True)
rankings.index = range(1, len(rankings) + 1)
rankings[["sailor_name", "school_slug", "elo", "races", "provisional"]].head(25)

## Chart the top skippers' rating trajectories

One point per regatta (the skipper's post-regatta rating) keeps the lines
readable; each skipper's biggest single-regatta gain and drop is annotated
with the regatta name.

In [ ]:
import matplotlib.pyplot as plt

hist = pd.DataFrame(history)
top_ids = rankings.head(6)["sailor_slug"].tolist()
top_names = names.loc[top_ids, "sailor_name"]
regatta_names = races.drop_duplicates("regatta_slug").set_index("regatta_slug")["regatta_name"]

plt.figure(figsize=(10, 6))
for sid in top_ids:
    s = hist[hist["sailor_slug"] == sid]
    # one point per regatta: the rating after that regatta's last race
    end = s.groupby("regatta_slug", sort=False).last().reset_index().sort_values("start_time")
    (line,) = plt.plot(
        end["start_time"], end["elo"], marker="o", markersize=4, label=top_names[sid]
    )

    # annotate the biggest single-regatta rise and fall with the regatta name
    end["delta"] = end["elo"].diff().fillna(end["elo"] - BASE_RATING)
    for _, row in (
        end.loc[[end["delta"].idxmax(), end["delta"].idxmin()]]
        .drop_duplicates("regatta_slug")
        .iterrows()
    ):
        plt.annotate(
            regatta_names.get(row["regatta_slug"], row["regatta_slug"]),
            (row["start_time"], row["elo"]),
            textcoords="offset points",
            xytext=(4, 6),
            fontsize=6.5,
            color=line.get_color(),
        )

plt.xlabel("Date")
plt.ylabel("Elo rating")
plt.title(f"{'+'.join(SEASONS)} ({PARTICIPANT}) - top skipper Elo trajectories")
plt.legend(loc="best", fontsize=8)
plt.tight_layout()
plt.show()

## Interactive version

The same trajectories with hover: every point shows the regatta, date, and
rating, so nothing needs a printed label. Plotly ships with Colab; install
it locally with `pip install plotly` if the import fails.

In [ ]:
import plotly.express as px

ends = (
    hist[hist["sailor_slug"].isin(top_ids)]
    .groupby(["sailor_slug", "regatta_slug"], sort=False)
    .last()
    .reset_index()
    .sort_values("start_time")
)
ends["sailor"] = ends["sailor_slug"].map(top_names)
ends["regatta"] = ends["regatta_slug"].map(regatta_names)

fig = px.line(
    ends,
    x="start_time",
    y="elo",
    color="sailor",
    markers=True,
    hover_name="regatta",
    hover_data={"start_time": "|%b %d", "elo": ":.1f", "sailor": False},
    title=f"{'+'.join(SEASONS)} ({PARTICIPANT}) - top skipper Elo trajectories",
    labels={"start_time": "Date", "elo": "Elo rating"},
)
fig.show()

---
**Simplifications vs. the production engine:** (`analytics/skipper_elo/engine.py`)
this notebook skips the weekly rank-timeline snapshots (used for the website's
subway chart), the cross-division A/B calibration offset (which estimates how
much B-division ratings run inflated relative to A, from sailors who raced
enough of both), and peak-rating/win-rate bookkeeping. The rating math itself —
grade lookup, K-factor scaling, expected-score formula, chronological
simultaneous pairwise updates — is unchanged.